In [24]:
import os
import random
import numpy as np
import pandas as pd

from sklearn.feature_selection import VarianceThreshold
from sklearn.model_selection import KFold
from sklearn.metrics import root_mean_squared_error

from catboost import CatBoostRegressor
from xgboost import XGBRegressor



def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)

seed_everything(42)

In [6]:
train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')
print(f"Исходные данные загружены. Train: {train.shape}, Test: {test.shape}")


Исходные данные загружены. Train: (751, 214), Test: (250, 211)


In [7]:
target_cols = ['IC50, mM', 'CC50, mM', 'SI']
feature_cols = [col for col in train.columns if col not in ["index"] + target_cols]


In [8]:
medians = train.groupby(feature_cols)[target_cols].transform('median')
train[target_cols] = medians
train_cleaned = train.drop_duplicates(subset=feature_cols, keep='first').reset_index(drop=True)
print(f"После объединения дубликатов молекул по медиане: {train_cleaned.shape}")


После объединения дубликатов молекул по медиане: (630, 214)


In [9]:
train_cleaned = train_cleaned.dropna(subset=target_cols).reset_index(drop=True)
print(f"После удаления строк с пустыми таргетами (NaN): {train_cleaned.shape}")


После удаления строк с пустыми таргетами (NaN): (628, 214)


In [10]:
selector = VarianceThreshold(threshold=0.0)
selector.fit(train_cleaned[feature_cols])
constant_features = [col for col, keep in zip(feature_cols, selector.get_support()) if not keep]
feature_cols = [col for col in feature_cols if col not in constant_features]
print(f"Удалено константных признаков: {len(constant_features)}. Осталось признаков: {len(feature_cols)}")


Удалено константных признаков: 18. Осталось признаков: 192


In [11]:
corr_matrix = train_cleaned[feature_cols].corr().abs()
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
high_corr_features = [column for column in upper_tri.columns if any(upper_tri[column] > 0.95)]
feature_cols = [col for col in feature_cols if col not in high_corr_features]
print(f"Удалено сильно коррелирующих признаков (>0.95): {len(high_corr_features)}. Итого признаков: {len(feature_cols)}")


Удалено сильно коррелирующих признаков (>0.95): 34. Итого признаков: 158


In [12]:
q25 = train_cleaned['SI'].quantile(0.25)
q75 = train_cleaned['SI'].quantile(0.75)
iqr = q75 - q25
upper_boundary = q75 + 3.0 * iqr
train_final = train_cleaned[train_cleaned['SI'] <= upper_boundary].reset_index(drop=True)
print(f"Граница выбросов по IQR для SI: {upper_boundary:.2f}. Удалено выбросов: {train_cleaned.shape[0] - train_final.shape[0]}")


Граница выбросов по IQR для SI: 48.59. Удалено выбросов: 49


In [15]:
train_final['fold'] = -1
kf = KFold(n_splits=5, shuffle=True, random_state=42)
for fold_idx, (train_idx, val_idx) in enumerate(kf.split(train_final)):
    train_final.loc[val_idx, 'fold'] = fold_idx

print(f"\n Очищенный датасет")
print(f"Размерность train_final: {train_final.shape}")
print(f"Количество признаков в feature_cols: {len(feature_cols)}")
print(f"Распределение строк по 5 фолдам:\n{train_final['fold'].value_counts().to_string()}")


 Очищенный датасет
Размерность train_final: (579, 215)
Количество признаков в feature_cols: 158
Распределение строк по 5 фолдам:
fold
1    116
0    116
2    116
3    116
4    115


C:\Users\Professional\AppData\Local\Temp\ipykernel_17752\3667698995.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_final['fold'] = -1


In [16]:
train_final = train_final.copy()


In [ ]:
models_ic50 = []
models_cc50 = []
models_si = []

train_final['oof_IC50'] = 0.0
train_final['oof_CC50'] = 0.0

for fold in range(5):
    train_idx = train_final[train_final['fold'] != fold].index
    val_idx = train_final[train_final['fold'] == fold].index

    model = CatBoostRegressor(iterations=1000, learning_rate=0.05, depth=6, random_seed=42, verbose=0)
    model.fit(train_final.loc[train_idx, feature_cols], train_final.loc[train_idx, 'IC50, mM'],
              eval_set=(train_final.loc[val_idx, feature_cols], train_final.loc[val_idx, 'IC50, mM']),
              early_stopping_rounds=100, verbose=False)

    train_final.loc[val_idx, 'oof_IC50'] = model.predict(train_final.loc[val_idx, feature_cols])
    models_ic50.append(model)

for fold in range(5):
    train_idx = train_final[train_final['fold'] != fold].index
    val_idx = train_final[train_final['fold'] == fold].index

    model = CatBoostRegressor(iterations=1000, learning_rate=0.05, depth=6, random_seed=42, verbose=0)
    model.fit(train_final.loc[train_idx, feature_cols], train_final.loc[train_idx, 'CC50, mM'],
              eval_set=(train_final.loc[val_idx, feature_cols], train_final.loc[val_idx, 'CC50, mM']),
              early_stopping_rounds=100, verbose=False)

    train_final.loc[val_idx, 'oof_CC50'] = model.predict(train_final.loc[val_idx, feature_cols])
    models_cc50.append(model)

feature_cols_si = feature_cols + ['oof_IC50', 'oof_CC50']
for fold in range(5):
    train_idx = train_final[train_final['fold'] != fold].index
    val_idx = train_final[train_final['fold'] == fold].index

    model = CatBoostRegressor(iterations=1000, learning_rate=0.05, depth=6, random_seed=42, verbose=0)
    model.fit(train_final.loc[train_idx, feature_cols_si], train_final.loc[train_idx, 'SI'],
              eval_set=(train_final.loc[val_idx, feature_cols_si], train_final.loc[val_idx, 'SI']),
              early_stopping_rounds=100, verbose=False)
    models_si.append(model)

test_preds_ic50 = np.zeros(len(test))
test_preds_cc50 = np.zeros(len(test))
test_preds_si = np.zeros(len(test))

for model in models_ic50:
    test_preds_ic50 += model.predict(test[feature_cols]) / 5

for model in models_cc50:
    test_preds_cc50 += model.predict(test[feature_cols]) / 5

test_meta = test.copy()
test_meta['oof_IC50'] = test_preds_ic50
test_meta['oof_CC50'] = test_preds_cc50

for model in models_si:
    test_preds_si += model.predict(test_meta[feature_cols_si]) / 5


submission = pd.DataFrame({
    'index': test['index'],
    'IC50': test_preds_ic50,
    'CC50': test_preds_cc50,
    'SI': test_preds_si
})

submission.to_csv('submission.csv', index=False)
print("Файл submission.csv создан. Формат:")
print(submission.head())
print(f"\nРазмерность файла: {submission.shape}")

Файл submission.csv создан. Формат:
   index        IC50        CC50        SI
0      0  173.109074  360.244931  8.202102
1      1  235.091709  379.337866  5.609795
2      2  158.658089  305.884217  8.568503
3      3  281.977935  396.177885  6.447149
4      4  207.574996  357.836030  4.830452

Размерность файла: (250, 4)


In [28]:
xgb_models_ic50 = []
xgb_models_cc50 = []
xgb_models_si = []

train_final['xgb_oof_IC50'] = 0.0
train_final['xgb_oof_CC50'] = 0.0

for fold in range(5):
    train_idx = train_final[train_final['fold'] != fold].index
    val_idx = train_final[train_final['fold'] == fold].index

    model = XGBRegressor(n_estimators=1000, learning_rate=0.05, max_depth=6, random_state=42, n_jobs=-1, early_stopping_rounds=100)
    model.fit(
        train_final.loc[train_idx, feature_cols], train_final.loc[train_idx, 'IC50, mM'],
        eval_set=[(train_final.loc[val_idx, feature_cols], train_final.loc[val_idx, 'IC50, mM'])],
        verbose=False
    )

    train_final.loc[val_idx, 'xgb_oof_IC50'] = model.predict(train_final.loc[val_idx, feature_cols])
    xgb_models_ic50.append(model)

for fold in range(5):
    train_idx = train_final[train_final['fold'] != fold].index
    val_idx = train_final[train_final['fold'] == fold].index

    model = XGBRegressor(n_estimators=1000, learning_rate=0.05, max_depth=6, random_state=42, n_jobs=-1, early_stopping_rounds=100)
    model.fit(
        train_final.loc[train_idx, feature_cols], train_final.loc[train_idx, 'CC50, mM'],
        eval_set=[(train_final.loc[val_idx, feature_cols], train_final.loc[val_idx, 'CC50, mM'])],
        verbose=False
    )

    train_final.loc[val_idx, 'xgb_oof_CC50'] = model.predict(train_final.loc[val_idx, feature_cols])
    xgb_models_cc50.append(model)

feature_cols_xgb_si = feature_cols + ['xgb_oof_IC50', 'xgb_oof_CC50']
xgb_oof_predictions_si = np.zeros(len(train_final))

for fold in range(5):
    train_idx = train_final[train_final['fold'] != fold].index
    val_idx = train_final[train_final['fold'] == fold].index

    model = XGBRegressor(n_estimators=1000, learning_rate=0.05, max_depth=6, random_state=42, n_jobs=-1, early_stopping_rounds=100)
    model.fit(
        train_final.loc[train_idx, feature_cols_xgb_si], train_final.loc[train_idx, 'SI'],
        eval_set=[(train_final.loc[val_idx, feature_cols_xgb_si], train_final.loc[val_idx, 'SI'])],
        verbose=False
    )
    xgb_oof_predictions_si[val_idx] = model.predict(train_final.loc[val_idx, feature_cols_xgb_si])
    xgb_models_si.append(model)

xgb_test_ic50 = np.zeros(len(test))
xgb_test_cc50 = np.zeros(len(test))
xgb_test_si = np.zeros(len(test))

for model in xgb_models_ic50:
    xgb_test_ic50 += model.predict(test[feature_cols]) / 5

for model in xgb_models_cc50:
    xgb_test_cc50 += model.predict(test[feature_cols]) / 5

test_meta_xgb = test.copy()
test_meta_xgb['xgb_oof_IC50'] = xgb_test_ic50
test_meta_xgb['xgb_oof_CC50'] = xgb_test_cc50

for model in xgb_models_si:
    xgb_test_si += model.predict(test_meta_xgb[feature_cols_xgb_si]) / 5

print("Обучение XGBoost завершено")
print(f"Локальный OOF RMSE для IC50 (XGB): {root_mean_squared_error(train_final['IC50, mM'], train_final['xgb_oof_IC50']):.4f}")
print(f"Локальный OOF RMSE для CC50 (XGB): {root_mean_squared_error(train_final['CC50, mM'], train_final['xgb_oof_CC50']):.4f}")
print(f"Локальный OOF RMSE для SI (XGB): {root_mean_squared_error(train_final['SI'], xgb_oof_predictions_si):.4f}")


Обучение XGBoost завершено
Локальный OOF RMSE для IC50 (XGB): 330.7210
Локальный OOF RMSE для CC50 (XGB): 451.3674
Локальный OOF RMSE для SI (XGB): 9.5953


In [29]:
blended_ic50 = (test_preds_ic50 + xgb_test_ic50) / 2
blended_cc50 = (test_preds_cc50 + xgb_test_cc50) / 2
blended_si = (test_preds_si + xgb_test_si) / 2
submission_blended = pd.DataFrame({
    'index': test['index'],
    'IC50': blended_ic50,
    'CC50': blended_cc50,
    'SI': blended_si
})

submission_blended.to_csv('submission_blended.csv', index=False)

print("Файл submission_blended.csv создан")
print(submission_blended.head())

Файл submission_blended.csv создан
   index        IC50        CC50        SI
0      0  183.376287  407.301821  8.574248
1      1  237.705301  390.244605  5.849062
2      2  153.684235  346.939434  8.095828
3      3  303.942306  438.748698  6.133202
4      4  198.019498  353.846655  5.372371
